# Dataframer: Richmond Latimore's Odyssey to Pandas DF

### [—————————————pipeline—————————————]
### »——raw—»—clean—»—normalize—»—DATAFRAME——»

Here are some transformation and frequencies for future exploratory analysis of Green's Odyssey.

Columns: author, year, title, book_num, text, num_lines, num_sentences, num_words, 


In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()

from collections import Counter
counter = Counter()

import re
import nltk

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/Users/debr/English-Homer') 
import bard_visualization as viz# This will apply the visualization settings
from bard_visualization import color_palette 

In [3]:
# Import my functions
sys.path.append('/Users/debr/English-Homer/functions') 
import e_nlp as e

Functions for NLP are live! use e.<function> to call them.
Download complete.


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# Cell display options
pd.set_option("display.max_colwidth", None)  # Prevent truncation of long values
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.expand_frame_repr", False)  # Prevent wrapping in DataFrames

In [5]:
# TO UPDATE
translator = "Lattimore" 

# Check Paths
filepath = f"/Users/debr/odysseys_en/Normalized_txts/Odyssey_{translator}_Normalized_v2.txt"

output_path = f"/Users/debr/English-Homer/dataframers_by_author/{translator}_DFed/"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

output_path_plots = f"{output_path}/plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)

# READING FILE TO extracted_lines
with open(filepath, 'r') as file:
    extracted_lines = file.readlines()

book_breaker = "BOOK"

books = e.list_into_books(extracted_lines, book_breaker)

# Verify the results
print(f"Found {len(books)} books")
for i, book in enumerate(books):
    print(f"Book {i+1} starts with: {book[0]}")
    print(f"Book {i+1} has {len(book)} lines")

Found 24 books
Book 1 starts with: Tell me, Muse, of the man of many ways, who was driven

Book 1 has 455 lines
Book 2 starts with: Now when the young Dawn showed again with her rosy fingers, the dear

Book 2 has 442 lines
Book 3 starts with: Helios, leaving behind the lovely standing waters, rose up

Book 3 has 510 lines
Book 4 starts with: They came into the cavernous hollow of Lakedaimon

Book 4 has 871 lines
Book 5 starts with: Now Dawn rose from her bed, where she lay by haughty Tithonos,

Book 5 has 507 lines
Book 6 starts with: So long-suffering great Odysseus slept in that place

Book 6 has 334 lines
Book 7 starts with: So long-suffering great Odysseus prayed, in that place,

Book 7 has 353 lines
Book 8 starts with: Then when the young Dawn showed again with her rosy fingers,

Book 8 has 600 lines
Book 9 starts with: Then resourceful Odysseus spoke in turn and answered him: ‘O great

Book 9 has 568 lines
Book 10 starts with: ‘We came next to the Aiolian island, whereAiolos

Boo

In [6]:
# Create a DataFrame from the list of books and bibliographic information
df = e.book_into_df(f"{translator}", "1965", "The Odyssey", books)

# New columns
df['num_lines'] = df['text'].apply(e.count_lines)
df['num_sentences'] = df['text'].apply(e.count_sentences)
df['num_words'] = df['text'].apply(e.count_words)

In [7]:
# removing \n from text column
lines = df['text'][0]
print("Before:", lines[:2])
lines_cl = e.remove_newline_character(lines)
print("After:", lines_cl[:2])
df['text'] = df['text'].apply(e.remove_newline_character)
print("df clean:", df['text'][0][:3])

Before: ['Tell me, Muse, of the man of many ways, who was driven\n', "far journeys, after he had sacked Troy's sacred citadel.\n"]
After: ['Tell me, Muse, of the man of many ways, who was driven', "far journeys, after he had sacked Troy's sacred citadel."]
df clean: ['Tell me, Muse, of the man of many ways, who was driven', "far journeys, after he had sacked Troy's sacred citadel.", 'Many were they whose cities he saw, whose minds he learned of,']


In [8]:
import sys
sys.path.append('/Users/debr/English-Homer/functions')
import e_nlp_pipeline as e_pipe

# Verify the module is imported correctly
print("Module imported:", e_pipe)
print("NLPPipeline class exists:", hasattr(e_pipe, "NLPPipeline"))

Pipiline live! use e.NLPPipeline(language="english")
Download complete.
Module imported: <module 'e_nlp_pipeline' from '/Users/debr/English-Homer/functions/e_nlp_pipeline.py'>
NLPPipeline class exists: True


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
# Process with default pipeline (lowercase -> tokenize -> remove punctuation -> remove stopwords)
# 1. Initialize the pipeline
pipe = e_pipe.NLPPipeline(language="english")
# 2. Customize
pipe.customize_stopwords(
    include={"one", "two", "three", "four", "five","six", "seven", "eight", "nine", "ten",
             "'", "n", "'and",
            },    
    exclude={""}
)
pipe.customize_punctuation(
    keep={"-", ""},  # Keep hyphens and apostrophes
    remove={"…", "—", "”", "’", "“", "‘"},  # Additional characters to remove
)
# 3. Text column: list->string
df['text'] = df['text'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# 4. Process the DataFrame's text column and add the tokens column
df = pipe.process_dataframe(df, 'text', "tokens")

Stopwords customized:
  Added: {'six', 'four', "'", 'ten', 'n', 'five', 'three', 'seven', 'eight', "'and", 'one', 'two', 'nine'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'”', '—', '‘', '“', '’', '…'}
  Punctuation to be removed: !"#$%&'()*+,./:;<=>?@[\]^_`{|}~—‘’“”…

Processing pipeline steps:
lowercase → tokenize → remove_punctuation → remove_stopwords


In [10]:
# Count the number of tokens in each book
df['tokens'].map(counter.update)
keywords = [word for word, count in counter.most_common(100)]
keywords_count = counter.most_common(100)
print(keywords)
print(keywords_count)

['odysseus', 'spoke', 'man', 'men', 'come', 'would', 'son', 'went', 'house', 'way', 'back', 'came', 'away', 'great', 'go', 'us', 'heart', 'telemachos', 'tell', 'ship', 'suitors', 'father', 'zeus', 'gods', 'let', 'made', 'said', 'since', 'time', 'palace', 'could', 'many', 'among', 'people', 'give', 'sea', 'upon', 'companions', 'put', 'answer', 'athene', 'hands', 'even', 'water', 'set', 'make', 'home', 'dear', 'turn', 'took', 'see', 'still', 'like', 'much', 'long', 'first', 'shall', 'country', 'gave', 'answered', 'wine', 'good', 'land', 'may', 'words', 'far', 'take', 'beside', 'old', 'place', 'stood', 'city', 'shining', 'god', 'must', 'also', 'told', 'others', 'mother', 'ships', 'women', 'never', 'spirit', 'hold', 'word', 'penelope', 'yet', 'achaians', 'mind', 'might', 'close', 'think', 'sleep', 'young', 'death', 'brought', 'know', 'inside', 'goddess', 'wife']
[('odysseus', 718), ('spoke', 684), ('man', 522), ('men', 509), ('come', 461), ('would', 408), ('son', 389), ('went', 380), ('hou

In [11]:
# Create directory OUTSIDE the project folder
output_filepath = f"/Users/debr/odysseys_en/Odyssey_dfs/Odyssey_{translator}_eda_END.csv"
os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

# save df to csv
df.to_csv(output_filepath, index=False)

print(f"Normalization complete. File saved to: {output_filepath}")

Normalization complete. File saved to: /Users/debr/odysseys_en/Odyssey_dfs/Odyssey_Lattimore_eda_END.csv
